In [ ]:
# imports
import numpy as np
import pandas as pd
from scipy import stats

from dual_pfc_funcs import jitter, getParams

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
p = getParams()
subjects = p['subjects']
color_map = p['color_map']
plot_sym = p['markers']
data_path = 'preprocessed_data/'

In [ ]:
# plot mean rsc across sessions
fname = data_path + 'within_across_rsc_means.pkl'
df = pd.read_pickle(fname,compression='gzip')

fig,ax = plt.subplots()

# add individual data points
for sub,sym in zip(subjects,plot_sym):
    filt = df['Subject']==sub
    ydata = 1.4+jitter(sum(filt),spacing=0.1)
    ax.scatter(df[filt]['AcrossAreaRscSimilar'], ydata, color=color_map['across'],alpha=0.5,s=12,marker=sym,edgecolors='none',label='Monkey {:s}'.format(sub[:2].title()))
    ydata = 0.4+jitter(sum(filt),spacing=0.1)
    ax.scatter(df[filt]['AcrossAreaRscOpposite'], ydata, color=color_map['across'],alpha=0.5,s=12,marker=sym,edgecolors='none')

mean_sim = np.mean(df['AcrossAreaRscSimilar'])
mean_opp = np.mean(df['AcrossAreaRscOpposite'])
sem_sim = stats.sem(df['AcrossAreaRscSimilar'])
sem_opp = stats.sem(df['AcrossAreaRscOpposite'])

ax.barh(1, mean_sim, color=color_map['across'], xerr=sem_sim, align='center', ecolor='black', edgecolor='black', height=0.4)
ax.barh(0, mean_opp, color=color_map['across'], xerr=sem_opp, align='center', ecolor='black', edgecolor='black', height=0.4)
ax.axvline(linewidth=1, color='k')

ax.set_xlabel('mean spike count correlation ($r_{sc}$)')
plt.yticks([1,0], ['similarly \n tuned pairs', 'oppositely \n tuned pairs'])
plt.legend(loc='lower right',markerscale=1.5)

if SAVE_FIG:
    pdf = PdfPages('figs/rsc_signal_tuning.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
print('Mean tuned rsc across sessions: similar tuning = {}, opposite tuning = {}'.format(mean_sim,mean_opp))

# statistics 
print()
alpha = 0.01 
print("rsc statistics across sessions:")
p = stats.ttest_1samp(df['AcrossAreaRscSimilar'], popmean=0,alternative='greater').pvalue
print("  similarly tuned > 0? {}, p = {:.6f}".format(p < alpha, p))
p = stats.ttest_1samp(df['AcrossAreaRscOpposite'], popmean=0,alternative='less').pvalue
print("  oppositely tuned < 0? {}, p = {:.6f}".format(p < alpha, p))